# 18 — Build Unified Matchup Feature Table

**Goal:** Produce a single artifact — `data/processed/ufc_matchup_features.csv` — that every downstream modeling notebook (19–23) consumes. This replaces the scattered 4-feature Z-delta approach used in notebooks 13/15/16/17 with a richer, leakage-free matrix.

**Feature groups included per fight (one row per fight, Fighter_A / Fighter_B perspective):**
1. **Static style** — GMM soft posteriors `pk3_*`, `pk5_*`, `Hybrid_Score_*`, heatmap lookup `p_hmap_A`, style Z-deltas (career).
2. **Autoencoder embeddings** — 3-D from `ufc_ae_embeddings.csv` (notebook 12).
3. **Rolling (leakage-free) pre-fight career rates** — `A_pre_*`, `B_pre_*` from `scripts/matchup_utils.build_rolling_profiles`.
4. **Form / tempo** — last-5 win rate, finish rate, control ratio; days since last fight.
5. **Ratings** — pre-fight Elo and Glicko-2 (mean + phi) walked chronologically.
6. **Physical** — height, reach, stance one-hots (from `raw_fighters.csv`).
7. **Context** — Weight_Class one-hots, event date.
8. **Vegas** (where available) — de-vigged `p_vegas_A`.

**Outputs:**
- `data/processed/ufc_matchup_features.csv` — one row per fight with all features + `split` column (train/val/test by event date, 60/20/20).
- `data/processed/ufc_fighter_profiles_rolling.csv` — for reuse when projecting future matchups.

**Reproducibility:** The split column is computed once here; notebooks 19–23 filter by `split` rather than re-deriving it. Walk-forward CV folds are available via `matchup_utils.walk_forward_folds` on the train+val portion.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('../scripts'))
import numpy as np
import pandas as pd
import matchup_utils as mu

DATA = '../data/processed'
RAW = '../data/raw'

fights = pd.read_csv(f'{DATA}/ufc_fight_stats_cleaned.csv')
styles = pd.read_csv(f'{DATA}/ufc_gmm_comparison.csv')
ae = pd.read_csv(f'{DATA}/ufc_ae_embeddings.csv')
phys = mu.load_physical(f'{RAW}/raw_fighters.csv')

ev_dates = mu.load_event_dates(f'{RAW}/raw_events.csv')
fights = fights.merge(
    ev_dates.rename(columns={'Event_Id': 'Event_Id_x'}),
    on='Event_Id_x', how='left',
)
print(f'Fight-row table: {len(fights)} rows | unique fights: {fights.Fight_Id.nunique()}')
print(f'Event dates parsed: {fights.Event_Date.notna().mean():.3f}')
print(f'Styles table: {len(styles)} fighters ({[c for c in styles.columns if c.startswith("pk")]})')
print(f'AE embeddings: {len(ae)} fighters | Physical: {len(phys)} fighters')


Fight-row table: 16964 rows | unique fights: 8482
Event dates parsed: 1.000
Styles table: 1190 fighters (['pk3_0', 'pk3_1', 'pk3_2', 'pk5_0', 'pk5_1', 'pk5_2', 'pk5_3', 'pk5_4'])
AE embeddings: 1190 fighters | Physical: 4441 fighters


In [2]:
rolling = mu.build_rolling_profiles(fights, event_dates_path=f'{RAW}/raw_events.csv')
rolling.to_csv(f'{DATA}/ufc_fighter_profiles_rolling.csv', index=False)
print('Rolling profiles:', rolling.shape)
# dedupe to one row per fight (Fighter_A perspective == first duplicate kept)
rolling_fight = rolling.drop_duplicates(subset='Fight_Id', keep='first').copy()
rolling_fight['Fight_Id'] = rolling_fight['Fight_Id'].astype(str)
rolling_fight = rolling_fight.rename(columns={'Fighter': 'Fighter_A', 'Opponent': 'Fighter_B'})
print('One row per fight:', rolling_fight.shape)


Rolling profiles: (8482, 59)
One row per fight: (8482, 59)


In [3]:
# Use one row per fight in chronological order for rating walks.
walk_df = rolling_fight[['Fight_Id', 'Event_Date', 'Fighter_A', 'Fighter_B', 'Win_A']].rename(
    columns={'Fighter_A': 'Fighter', 'Fighter_B': 'Opponent'}
).sort_values(['Event_Date', 'Fight_Id']).reset_index(drop=True)
elo_df = mu.walk_elo(walk_df)
glicko_df = mu.walk_glicko(walk_df)
print('Elo/Glicko rows:', len(elo_df), len(glicko_df))
print(elo_df.head(3))
print(glicko_df.head(3))


Elo/Glicko rows: 8482 8482
           Fight_Id  Elo_A_pre  Elo_B_pre  elo_diff  elo_p_A  \
0  00835554f95fa911     1500.0     1500.0       0.0      0.5   
1  17ee4caf06981b81     1500.0     1500.0       0.0      0.5   
2  3b020d4914b44fc8     1500.0     1500.0       0.0      0.5   

   elo_favorite_is_A  
0              False  
1              False  
2              False  
           Fight_Id  Glicko_A_pre  Glicko_B_pre  Glicko_A_phi  Glicko_B_phi  \
0  00835554f95fa911        1500.0        1500.0         350.0         350.0   
1  17ee4caf06981b81        1500.0        1500.0         350.0         350.0   
2  3b020d4914b44fc8        1500.0        1500.0         350.0         350.0   

   glicko_diff  glicko_p_A  
0          0.0         0.5  
1          0.0         0.5  
2          0.0         0.5  


In [4]:
PK3 = [f'pk3_{i}' for i in range(3)]
PK5 = [f'pk5_{i}' for i in range(5)]
STATIC_Z = ['Sig_Str_PM_Z', 'Takedown_Att_PM_Z', 'Sub_Att_PM_Z', 'Control_Ratio_Z']
HYB = ['Hybrid_Score_k3', 'Hybrid_Score_k5']
style_cols = ['Fighter'] + PK3 + PK5 + STATIC_Z + HYB + ['Cluster_k5']
s = styles[style_cols].copy()
emb = ae.rename(columns={'Emb_1': 'ae1', 'Emb_2': 'ae2', 'Emb_3': 'ae3'})

# Attach per-corner style features
def attach(corner: str, df: pd.DataFrame) -> pd.DataFrame:
    renamed = {c: f'{corner}_{c}' for c in s.columns if c != 'Fighter'}
    renamed_e = {c: f'{corner}_{c}' for c in emb.columns if c != 'Fighter'}
    renamed_p = {c: f'{corner}_{c}' for c in phys.columns if c != 'Fighter'}
    out = df.merge(s.rename(columns=renamed), left_on=f'Fighter_{corner}', right_on='Fighter', how='left').drop(columns=['Fighter'])
    out = out.merge(emb.rename(columns=renamed_e), left_on=f'Fighter_{corner}', right_on='Fighter', how='left').drop(columns=['Fighter'])
    out = out.merge(phys.rename(columns=renamed_p), left_on=f'Fighter_{corner}', right_on='Fighter', how='left').drop(columns=['Fighter'])
    return out

base = rolling_fight.copy()
base = attach('A', base)
base = attach('B', base)
print('After style + AE + physical attach:', base.shape)


After style + AE + physical attach: (8482, 101)


In [5]:
# Delta and mean/tempo features for numerical style & physical fields.
DELTA_SRC = PK3 + PK5 + STATIC_Z + HYB + ['ae1', 'ae2', 'ae3', 'Height_In', 'Reach_In']
for c in DELTA_SRC:
    a, b = f'A_{c}', f'B_{c}'
    if a in base.columns and b in base.columns:
        base[f'delta_{c}'] = base[a] - base[b]
        base[f'mean_{c}'] = (base[a] + base[b]) / 2  # joint tempo (useful for dynamics)

# Rolling-rate deltas (pre-fight leakage-free; use these instead of Z-deltas when possible)
ROLL = ['pre_sig_str_pm', 'pre_sig_str_att_pm', 'pre_sig_str_absorbed_pm',
        'pre_td_landed_pm', 'pre_td_att_pm', 'pre_td_absorbed_pm',
        'pre_sub_att_pm', 'pre_control_ratio', 'pre_kd_for_pm', 'pre_kd_against_pm',
        'pre_win_rate', 'pre_ko_rate_for', 'pre_sub_rate_for',
        'pre_ko_rate_against', 'pre_sub_rate_against',
        'pre_ground_share', 'pre_clinch_share', 'pre_distance_share',
        'pre_sig_str_acc', 'pre_td_acc',
        'recent5_win_rate', 'recent5_finish_rate', 'recent5_ctrl_ratio']
for c in ROLL:
    a, b = f'A_{c}', f'B_{c}'
    if a in base.columns and b in base.columns:
        base[f'delta_{c}'] = base[a] - base[b]
        base[f'mean_{c}'] = (base[a] + base[b]) / 2

# Stance one-hots (per corner + matchup indicator for southpaw vs orthodox)
for corner in ('A', 'B'):
    d = pd.get_dummies(base[f'{corner}_Stance'].fillna('Unknown'), prefix=f'{corner}_stance').astype(float)
    base = pd.concat([base, d], axis=1)
base['southpaw_vs_orthodox'] = (
    ((base['A_Stance'] == 'Southpaw') & (base['B_Stance'] == 'Orthodox'))
    | ((base['A_Stance'] == 'Orthodox') & (base['B_Stance'] == 'Southpaw'))
).astype(float)

# Days-since-last-fight delta (ring rust asymmetry)
base['delta_days_since_last'] = base['days_since_last_A'].fillna(365) - base['days_since_last_B'].fillna(365)
base['mean_days_since_last'] = (base['days_since_last_A'].fillna(365) + base['days_since_last_B'].fillna(365)) / 2

# Weight-class one-hots (explicit inclusion: division-level base rates matter for dynamics)
base, WC_COLS = mu.add_weight_class_dummies(base, col='Weight_Class')
print(f'Weight-class dummies: {len(WC_COLS)} | total cols now: {base.shape[1]}')


Weight-class dummies: 15 | total cols now: 215


In [6]:
# Build the same empirical heatmap used in NB 11 from the styled matchup set.
styled = fights.merge(
    styles[['Fighter', 'Cluster_k5']], on='Fighter', how='inner'
).rename(columns={'Cluster_k5': 'Style_A'})
styled = styled.merge(
    styles[['Fighter', 'Cluster_k5']].rename(columns={'Fighter': 'Opponent'}),
    on='Opponent', how='inner',
).rename(columns={'Cluster_k5': 'Style_B'})
styled['Win_A'] = styled['Won'].astype(int)
hmap = styled.pivot_table(index='Style_A', columns='Style_B', values='Win_A', aggfunc='mean')
print('Heatmap (NB 11 replica):'); print(hmap.round(3))

def hmap_p_a(row):
    ca, cb = row.get('A_Cluster_k5'), row.get('B_Cluster_k5')
    if pd.isna(ca) or pd.isna(cb):
        return np.nan
    try:
        return float(hmap.loc[int(ca), int(cb)])
    except Exception:
        return np.nan

base['p_hmap_A'] = base.apply(hmap_p_a, axis=1)
print('heatmap-lookup non-null:', base['p_hmap_A'].notna().mean())


Heatmap (NB 11 replica):
Style_B      0      1      2      3      4
Style_A                                   
0        0.475  0.421  0.614  0.429  0.497
1        0.579  0.491  0.541  0.505  0.467
2        0.364  0.452  0.500  0.453  0.481
3        0.566  0.475  0.529  0.489  0.475
4        0.490  0.518  0.496  0.507  0.488


heatmap-lookup non-null: 0.653383635934921


In [7]:
base = base.merge(elo_df, on='Fight_Id', how='left')
base = base.merge(glicko_df, on='Fight_Id', how='left')
# Glicko uncertainty: bigger phi = less confident rating (feature for upset models)
base['glicko_phi_sum'] = base['Glicko_A_phi'].fillna(350) + base['Glicko_B_phi'].fillna(350)

# Optional Vegas de-vigged odds (NB 14). Skip gracefully if missing.
odds_path = f'{DATA}/ufc_fight_odds_clean.csv'
if os.path.exists(odds_path):
    odds = pd.read_csv(odds_path)
    odds['Fight_Id'] = odds['Fight_Id'].astype(str)
    def norm(x):
        return str(x).strip().lower() if pd.notna(x) else np.nan
    odds['n1'] = odds['fighter_1'].map(norm)
    odds['n2'] = odds['fighter_2'].map(norm)
    base['nA'] = base['Fighter_A'].map(norm)
    base['nB'] = base['Fighter_B'].map(norm)
    base = base.merge(
        odds[['Fight_Id', 'n1', 'n2', 'p_fighter_1', 'p_fighter_2']],
        on='Fight_Id', how='left',
    )
    # Map de-vigged prob onto A's corner
    p_vegas_A = np.where(
        base['nA'] == base['n1'], base['p_fighter_1'],
        np.where(base['nA'] == base['n2'], base['p_fighter_2'], np.nan),
    )
    base['p_vegas_A'] = p_vegas_A.astype(float)
    base['has_vegas'] = base['p_vegas_A'].notna().astype(int)
    base = base.drop(columns=['n1', 'n2'])
    print(f'Vegas coverage: {base.has_vegas.mean():.3f} of fights')
else:
    base['p_vegas_A'] = np.nan
    base['has_vegas'] = 0
    print('No Vegas odds file (run NB 14).')


Vegas coverage: 0.721 of fights


In [8]:
# Dynamics targets attached here so NB 21 can consume the same table.
method_map = dict(zip(fights['Fight_Id'].astype(str), fights['Method']))
base['method_6'] = [mu.method_bucket(method_map.get(str(fid), ''), int(w))
                   for fid, w in zip(base['Fight_Id'], base['Win_A'])]
base['method_3'] = [mu.method_3way(method_map.get(str(fid), ''))
                   for fid in base['Fight_Id']]
base['y_finish'] = base['method_3'].isin(['ko', 'sub']).astype(int)
base['y_finish_valid'] = base['method_3'].notna().astype(int)

# Regression targets (pooled over both corners of the fight, so they describe the whole bout)
fight_agg = fights.groupby('Fight_Id').agg(
    tot_seconds=('total_fight_seconds', 'max'),
    tot_sig_str=('Sig_Str_Landed', 'sum'),
    tot_td=('Takedowns_Landed', 'sum'),
    tot_sub_att=('Sub_Attempts', 'sum'),
    tot_ctrl=('Control_Seconds', 'sum'),
    tot_kd=('Knockdowns', 'sum'),
    tot_ground=('Ground_Strikes_Landed', 'sum'),
    tot_clinch=('Clinch_Strikes_Landed', 'sum'),
    tot_distance=('Distance_Strikes_Landed', 'sum'),
).reset_index()
fight_agg['Fight_Id'] = fight_agg['Fight_Id'].astype(str)
base = base.merge(fight_agg, on='Fight_Id', how='left')
base['ground_share_target'] = np.where(base['tot_sig_str'] > 0, base['tot_ground'] / base['tot_sig_str'], np.nan)

# Event-ordered 60/20/20 split
splits = mu.event_ordered_split(base['Event_Id_x'], base['Event_Date'])
base['split'] = np.where(splits['train'], 'train', np.where(splits['val'], 'val', 'test'))
print('Split counts:', base.split.value_counts().to_dict())

# Gender flag: the UFCStats scrape distinguishes women's divisions by a
# "Women's ..." prefix; everything else is men's.  Z-scoring is already done
# within Weight_Class (NB 05), so this column is purely for downstream slicing
# and reporting (NB 20 groupwise calibration, NB 25 champion validation, etc.).
base['Gender'] = np.where(
    base['Weight_Class'].fillna('').str.startswith("Women"), 'F', 'M'
)
print('Gender counts:', base['Gender'].value_counts().to_dict())


Split counts: {'train': 4846, 'test': 1850, 'val': 1786}
Gender counts: {'M': 7594, 'F': 888}


In [9]:
out_path = f'{DATA}/ufc_matchup_features.csv'
base.to_csv(out_path, index=False)
print(f'Wrote {out_path}  shape={base.shape}')

def expand(names):
    out = []
    for n in names:
        out += [f'A_{n}', f'B_{n}', f'delta_{n}', f'mean_{n}']
    return out

groups = {
    'style_gmm_probs': expand(PK3 + PK5),
    'hybrid': expand(HYB),
    'style_z': expand(STATIC_Z),
    'ae': expand(['ae1', 'ae2', 'ae3']),
    'rolling_rates': expand(ROLL),
    'physical': expand(['Height_In', 'Reach_In']),
    'heatmap': ['p_hmap_A'],
    'elo_glicko': ['Elo_A_pre', 'Elo_B_pre', 'elo_diff', 'elo_p_A',
                   'Glicko_A_pre', 'Glicko_B_pre', 'glicko_diff', 'glicko_p_A', 'glicko_phi_sum'],
    'vegas': ['p_vegas_A', 'has_vegas'],
    'context_days': ['days_since_last_A', 'days_since_last_B', 'delta_days_since_last', 'mean_days_since_last'],
    'stance_wc': [c for c in base.columns if c.startswith(('A_stance_', 'B_stance_', 'wc_'))] + ['southpaw_vs_orthodox'],
}
for k, v in groups.items():
    present = [c for c in v if c in base.columns]
    print(f'  {k}: {len(present)} cols')

# Persist group mapping for downstream notebooks
import json
with open(f'{DATA}/ufc_feature_groups.json', 'w') as f:
    json.dump({k: [c for c in v if c in base.columns] for k, v in groups.items()}, f, indent=2)
print('Feature groups saved to ufc_feature_groups.json')


Wrote ../data/processed/ufc_matchup_features.csv  shape=(8482, 250)
  style_gmm_probs: 32 cols
  hybrid: 8 cols
  style_z: 16 cols
  ae: 12 cols
  rolling_rates: 92 cols
  physical: 8 cols
  heatmap: 1 cols
  elo_glicko: 9 cols
  vegas: 2 cols
  context_days: 4 cols
  stance_wc: 28 cols
Feature groups saved to ufc_feature_groups.json


### Takeaway
- Every downstream notebook should now load **`ufc_matchup_features.csv`** and filter by the `split` column rather than re-deriving a train/test split. This makes metrics directly comparable across NB 19–23.
- **Rolling vs static** features: for win/loss models use `delta_pre_*` (leakage-free). For style discovery / descriptive heatmaps keep the static Z-deltas.
- **Weight-class one-hots** (`wc_*`) are included so tree and linear models can learn division-specific base rates (e.g. bantamweight finishes more than welterweight).
- **Vegas prob** is present when odds exist and is used both as a baseline *and* as an input feature in notebook 22's residual upset model.